# Immigration Enforcement in Texas Network and Routing Analysis

In [1]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import os
import pandas as pd
from multiprocessing import Pool, Process, Queue, Lock, Value, Array, Manager

In [2]:
os.getcwd()

'/home/jovyan/work/GGIS 407/Final Project'

In [3]:
os.listdir("/home/jovyan/work/GGIS 407/Final Project/data")

['facilities.csv',
 'geojson-counties-fips.json',
 'EOIR scraper.ipynb',
 'pgdata',
 '.ipynb_checkpoints',
 'ice-aor-county-shp.shx',
 'tx_287g_counties_gdf.geojson',
 'immigration_court_administrative_control.csv',
 'ice-aor-county-shp.prj',
 'participating_pendingAgencies02122026pm.csv',
 'courts_gdf.geojson',
 'download_contracts.csv',
 'processed',
 'ice-aor-county-shp.shp',
 'texas_immigration_judges.csv',
 'ice-aor-county-shp.dbf',
 'EOIR_complete_gdf.geojson',
 'EFF_towers_01_26.csv']

## Initialize PostGIS Database and Import Datasets

In [4]:
os.environ["PGDATA"] = "/home/jovyan/work/GGIS 407/Final Project/data/pgdata"
!pg_ctl initdb
!pg_ctl start
%reload_ext sql
!createdb tx_immigration_enforcement
%sql postgresql://localhost:5432/tx_immigration_enforcement

The files belonging to this database system will be owned by user "jovyan".
This user must also own the server process.

The database cluster will be initialized with locale "en_US.UTF-8".
The default database encoding has accordingly been set to "UTF8".
The default text search configuration will be set to "english".

Data page checksums are disabled.

initdb: error: directory "/home/jovyan/work/GGIS 407/Final Project/data/pgdata" exists but is not empty
If you want to create a new database system, either remove or empty
the directory "/home/jovyan/work/GGIS 407/Final Project/data/pgdata" or run initdb
with an argument other than "/home/jovyan/work/GGIS 407/Final Project/data/pgdata".
pg_ctl: database system initialization failed
pg_ctl: another server might be running; trying to start server anyway
waiting for server to start....2026-03-06 19:22:02.397 UTC [2954] FATAL:  lock file "postmaster.pid" already exists
2026-03-06 19:22:02.397 UTC [2954] HINT:  Is another postmaster (PID 1964

In [5]:
%%sql
drop extension if exists postgis cascade;
create extension postgis;

 * postgresql://localhost:5432/tx_immigration_enforcement
Done.
Done.


[]

In [6]:
!shp2pgsql -s 4269 -d -I "/home/jovyan/work/GGIS 407/Final Project/data/ice-aor-county-shp.shp" public.ice_aor | psql -q -d tx_immigration_enforcement

Field aland is an FTDouble with width 24 and precision 15
Field awater is an FTDouble with width 24 and precision 15
Shapefile type: Polygon
Postgis type: MULTIPOLYGON[2]
ERROR:  column not found in geometry_columns table
CONTEXT:  PL/pgSQL function dropgeometrycolumn(character varying,character varying,character varying,character varying) line 33 at RAISE
SQL statement "SELECT public.DropGeometryColumn('',$1,$2,$3)"
PL/pgSQL function dropgeometrycolumn(character varying,character varying,character varying) line 5 at SQL statement
                    addgeometrycolumn                    
---------------------------------------------------------
 public.ice_aor.geom SRID:4269 TYPE:MULTIPOLYGON DIMS:2 
(1 row)



In [7]:
# Import GeoJSON files into PostgreSQL database
!ogr2ogr -f "PostgreSQL" PG:"dbname=tx_immigration_enforcement host=localhost port=5432" \
    "/home/jovyan/work/GGIS 407/Final Project/data/processed/courts_gdf.geojson" \
    -nln public.courts \
    -t_srs EPSG:4269 \
    -overwrite

!ogr2ogr -f "PostgreSQL" PG:"dbname=tx_immigration_enforcement host=localhost port=5432" \
    "/home/jovyan/work/GGIS 407/Final Project/data/processed/EOIR_complete_gdf.geojson" \
    -nln public.detention_centers \
    -t_srs EPSG:4269 \
    -overwrite

!ogr2ogr -f "PostgreSQL" PG:"dbname=tx_immigration_enforcement host=localhost port=5432" \
    "/home/jovyan/work/GGIS 407/Final Project/data/processed/tx_287g_counties_gdf.geojson" \
    -nln public.counties_287g \
    -t_srs EPSG:4269 \
    -overwrite

In [8]:
%sql \d

 * postgresql://localhost:5432/tx_immigration_enforcement
14 rows affected.


Schema,Name,Type,Owner
public,aor_to_counties,table,jovyan
public,counties_287g,table,jovyan
public,counties_287g_ogc_fid_seq,sequence,jovyan
public,courts,table,jovyan
public,courts_ogc_fid_seq,sequence,jovyan
public,detention_centers,table,jovyan
public,detention_centers_ogc_fid_seq,sequence,jovyan
public,detention_facilities,table,jovyan
public,geography_columns,view,jovyan
public,geometry_columns,view,jovyan


In [9]:
%%sql
SELECT column_name, data_type 
FROM information_schema.columns 
WHERE table_name = 'detention_centers';

 * postgresql://localhost:5432/tx_immigration_enforcement
9 rows affected.


column_name,data_type
ogc_fid,integer
facility_name,character varying
facility_type,character varying
administrative_control_court,character varying
court_address,character varying
judge_name,character varying
latitude,double precision
longitude,double precision
wkb_geometry,USER-DEFINED


In [10]:
%%sql
select *
from geometry_columns

 * postgresql://localhost:5432/tx_immigration_enforcement
4 rows affected.


f_table_catalog,f_table_schema,f_table_name,f_geometry_column,coord_dimension,srid,type
tx_immigration_enforcement,public,ice_aor,geom,2,4269,MULTIPOLYGON
tx_immigration_enforcement,public,courts,wkb_geometry,2,4269,POINT
tx_immigration_enforcement,public,detention_centers,wkb_geometry,2,4269,POINT
tx_immigration_enforcement,public,counties_287g,wkb_geometry,2,4269,GEOMETRY


## Query Operations

In [11]:
%%sql
drop table if exists aor_to_counties;
create table aor_to_counties as
select 
    name AS county, 
    aor_nam, 
    offc_nm, 
    geom, 
    aland, 
    awater
from ice_aor
where state_n = 'Texas'
order by aor_nam

 * postgresql://localhost:5432/tx_immigration_enforcement
Done.
254 rows affected.


[]

### Create table: `detention_facilities`

In [12]:
%%sql 
drop table if exists detention_facilities;
create table detention_facilities as
select 
    facility_name, 
    facility_type, 
    administrative_control_court, 
    court_address, 
    ST_AsText(wkb_geometry) as wkb_geometry
FROM detention_centers
WHERE facility_name IS NOT NULL

 * postgresql://localhost:5432/tx_immigration_enforcement
Done.
55 rows affected.


[]

In [13]:
%%sql
with tbl as (
    select 
        administrative_control_court, 
        count(facility_name) as total_facility_count
    from detention_facilities 
    group by administrative_control_court
)

select 
    d.administrative_control_court, 
    d.facility_type, 
    count(d.facility_name) as facility_type_count, 
    tbl.total_facility_count
from detention_facilities as d
left join tbl on d.administrative_control_court = tbl.administrative_control_court 
group by d.administrative_control_court, d.facility_type, tbl.total_facility_count
order by d.administrative_control_court

 * postgresql://localhost:5432/tx_immigration_enforcement
36 rows affected.


administrative_control_court,facility_type,facility_type_count,total_facility_count
Conroe Immigration Court,Detention Facility,5,7
Conroe Immigration Court,Processing Center,1,7
Conroe Immigration Court,Service Processing Center,1,7
Dallas Immigration Court,DHS Office,1,3
Dallas Immigration Court,Juvenile Facility,2,3
El Paso Immigration Court,Detention Facility,1,4
El Paso Immigration Court,DHS Office,1,4
El Paso Immigration Court,Port of Entry,1,4
El Paso Immigration Court,Processing Center,1,4
El Paso SPC Immigration Court,Detention Facility,1,3


### Create table: `judge_details`

In [14]:
%%sql 
select 
    administrative_control_court, 
    court_address, 
    total_judges,
    ST_AsText(wkb_geometry), 
    wkb_geometry
FROM courts 

 * postgresql://localhost:5432/tx_immigration_enforcement
12 rows affected.


administrative_control_court,court_address,total_judges,st_astext,wkb_geometry
Port Isabel Immigration Court,"27991 Buena Vista Blvd., Los Fresnos, TX 78566",4,POINT(-97.358682873261 26.158641827366),0101000020AD100000A59202A9F45658C08D5034C09C283A40
El Paso SPC Immigration Court,"8915 Montana Ave., Suite 100, El Paso, TX 79925",5,POINT(-106.369208351881 31.793762008564),0101000020AD1000004B2F111CA1975AC0F396ABFC33CB3F40
Harlingen Immigration Court,"2009 West Jefferson Ave., Suite 300, Harlingen, TX 78550",5,POINT(-97.718684038948 26.194727004596),0101000020AD10000016DC56EBFE6D58C04B6304A1D9313A40
Laredo Immigration Court,"1406 Jacaman Rd Suite B, Laredo, Tx 78041",6,POINT(-99.471972072251 27.562795340811),0101000020AD10000061BC59CA34DE58C0DA1FFF5A13903B40
Pearsall Immigration Court,"566 Veterans Drive, Pearsall, TX 78061",7,POINT(-99.120291973986 28.896570461288),0101000020AD1000006F8F1BDDB2C758C0A6CA49A485E53C40
El Paso Immigration Court,"700 East San Antonio Avenue Suite 750, El Paso, TX 79901",8,POINT(-106.482273268738 31.7592035475),0101000020AD100000BE3DB390DD9E5AC00A85E7295BC23F40
Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",10,POINT(-95.442699014881 30.337100979098),0101000020AD100000A9B83F2E55DC57C0EDACF03F4C563E40
Dallas Immigration Court,"1100 Commerce St. Room 1060, Dallas, TX 75242",11,POINT(-96.802110689134 32.778890064349),0101000020AD100000906612C8553358C076C76CABB2634040
Houston - Jefferson Street Immigration Court,"500 Jefferson Street, Suite 300, Houston TX 77002",13,POINT(-95.373800357512 29.75292813487),0101000020AD100000D1AF5558ECD757C04381F3E5BFC03D40
San Antonio Immigration Court,"800 Dolorosa Street Suite 300, San Antonio, TX 78207",13,POINT(-98.498824480222 29.423911194049),0101000020AD100000DA3F83BDEC9F58C049D9AA71856C3D40


In [15]:
%%sql 
drop table if exists judge_details;
create table judge_details as
select 
    judge_name, 
    administrative_control_court, 
    court_address, 
    ST_AsText(wkb_geometry) as geom_astext, 
    wkb_geometry
FROM detention_centers
WHERE judge_name IS NOT NULL

 * postgresql://localhost:5432/tx_immigration_enforcement
Done.
131 rows affected.


[]

In [16]:
%%sql
select 
    administrative_control_court, 
    count(judge_name) as judge_count, 
    geom_astext
from judge_details
group by administrative_control_court, geom_astext
order by judge_count desc

 * postgresql://localhost:5432/tx_immigration_enforcement
13 rows affected.


administrative_control_court,judge_count,geom_astext
Fort Worth IAC Immigration Court,18,None
Houston - Greenspoint Park Immigration Court,17,POINT(-95.402484068369 29.945864458543)
Houston - S. Gessner Road Immigration Court,14,POINT(-95.528444794718 29.686902919638)
San Antonio Immigration Court,13,POINT(-98.498824480222 29.423911194049)
Houston - Jefferson Street Immigration Court,13,POINT(-95.373800357512 29.75292813487)
Dallas Immigration Court,11,POINT(-96.802110689134 32.778890064349)
Conroe Immigration Court,10,POINT(-95.442699014881 30.337100979098)
El Paso Immigration Court,8,POINT(-106.482273268738 31.7592035475)
Pearsall Immigration Court,7,POINT(-99.120291973986 28.896570461288)
Laredo Immigration Court,6,POINT(-99.471972072251 27.562795340811)


### Create table: `county_287g`

In [17]:
%%sql 
select *
from counties_287g
limit 5

 * postgresql://localhost:5432/tx_immigration_enforcement
5 rows affected.


ogc_fid,fips,county,censusarea,state,law enforcement agency,type,signed,moa,addendum,jail enforcement model,task force model,warrant service officer,wkb_geometry
1,48001,Anderson,1062.602,TEXAS,Anderson County Sheriff's Office,County,2025-06-18 00:00:00+00:00,link,None,0,0,1,0103000020AD1000000100000013000000637C98BD6CDB57C0302AA913D00A404083A5BA8097DC57C0FDBB3E73D6D73F406F10AD156DDA57C0850662D9CCD53F409ED1562591D057C09FE6E445269C3F40D28A6F287CD157C09A417C60C7973F40732D5A80B6E957C06344A2D0B28A3F40A7E7DD5850EF57C0425A63D009813F40D11F9A7972ED57C0BFD7101C979D3F4003098A1F63F257C04356B77A4E9E3F4000581D39D2F257C0C5387F130AA93F4067D2A6EA1EF757C03D9AEAC9FCAF3F40CE16105A0FF857C0A70705A568C13F4044E048A0C1FE57C091B75CFDD8C83F408544DAC69FFF57C0FBE59315C3DD3F4029CE5147C70158C040F9BB77D4E03F40E6913F18780158C05F45460724F53F4086AE44A0FA0358C04DDBBFB2D2F43F40F2EB87D8600358C0DF1AD82AC1004040637C98BD6CDB57C0302AA913D00A4040
2,48005,Angelina,797.778,TEXAS,Angelina County Sheriff's Office,County,2025-06-18 00:00:00+00:00,link,None,0,0,1,0103000020AD10000001000000240000006C96CB46E79457C02BA5677A89393F40F72004E44B8857C09A25016A6A193F401ADD41EC4C9D57C0431B800D88083F40D5CDC5DFF6A357C0A708707A170F3F40B950F9D7F2B557C02750C42286253F406269E04735BA57C08DD2A57F49563F401EFE9AAC51BD57C05ED72FD80D633F40AE122C0E67BD57C00BF0DDE68D633F404568041BD7BD57C01C08C90226643F4089D349B6BABD57C0D2AB014A43653F40DA1F28B7EDBD57C0B709F7CABC653F40663046240ABE57C06D3656629E653F4072C0AE264FBE57C01B48179B56663F40B4AB90F293BE57C0CFA44DD53D663F4004ABEAE577BE57C0F4FDD478E9663F4044183F8D7BBE57C039B4C876BE673F404EB857E6ADBE57C05DFE43FAED673F4046B41D5377BE57C0E62329E961683F40054F2157EABE57C03543AA285E693F40D368723106BF57C01973D712F2693F40FF5C34643CBF57C00CE8853B176A3F408FA850DD5CBF57C0FA9AE5B2D1693F40F86D88F19ABF57C0605969520A6A3F40492C29779FBF57C0C91F0C3CF76A3F40E272BC02D1BF57C07BA01518B26A3F4021956247E3BF57C00803CFBD876B3F40F0366F9C14C057C05A65A6B4FE6A3F400DFE7E315BC057C0CA332F87DD6B3F40BA66F2CD36C057C085949F54FB6C3F404644317903C057C02C47C8409E6D3F40D0807A336AB757C038BD8BF7E3863F400C923EADA2AE57C082E15CC30C753F40E7FEEA71DFA257C05131CEDF846E3F4053094FE8F5A157C06A6B44300E663F40F33B4D66BC9F57C04D124BCADD673F406C96CB46E79457C02BA5677A89393F40
3,48011,Armstrong,909.109,TEXAS,Armstrong County Sheriff's Office,County,2026-01-26 00:00:00+00:00,link,None,0,1,0,0103000020AD1000000100000007000000540262122E5E59C05A80B6D5AC5F4140DCF126BF456859C08DF161F6B25F4140ED0E2906486859C09415C3D5016041409A95ED43DE6759C0BB2BBB60709741400951BEA0854559C08FA50F5D5097414043C9E4D4CE4559C071AE6186C65F4140540262122E5E59C05A80B6D5AC5F4140
4,48013,Atascosa,1219.544,TEXAS,Atascosa County Sheriff's Office,County,2025-05-08 00:00:00+00:00,link,None,0,1,0,0103000020AD100000010000000A000000F1129CFA40B358C0CD9541B5C1A53C403A234A7B83B358C04B72C0AE26173D40F981AB3C81B358C00C569C6A2D403D40F19E03CB119A58C08C84B69C4B1D3D4024B55032398C58C07B2E5393E0E13C40336DFFCA4A8658C0B29E5A7D75C93C405B09DD25719558C0FDA19927D79C3C40DEE7F868719558C07DD0B359F5A53C404A44F81741B358C02C8194D8B5A53C40F1129CFA40B358C0CD9541B5C1A53C40
5,48015,Austin,646.508,TEXAS,Austin County Sheriff's Office,County,2025-04-16 00:00:00+00:00,link,None,0,0,1,0103000020AD100000010000001000000007B13385CE2758C0CEC47421560B3E40A56ABB09BE1258C0AA8251499D183E40FCE07CEA580958C0F1643733FA113E40F48AA71E690558C0E3FF8EA850013E4001DD9733DB0758C04EB4AB90F2F73D4019E59997C30758C02A711DE38AD73D407BF65CA6260358C08692C9A99DCD3D400CCEE0EF170258C0C651B9895ABA3D408F537424970158C0B682A625569A3D40340EF5BBB00558C074603942069A3D40EE3F321D3A0B58C032022A1C41A23D407711A628971058C0E6E61BD13DAB3D40719010E50B1658C05DDF878384D43D408A3F8A3A731A58C0C9B08A3732D33D4099F4F752782458C0AB949EE925F63D4007B13385CE2758C0CEC47421560B3E40


## Import Area of Responsibility Shapefiles from Deportation Data Project at UC Berkeley

In [18]:
aor = gpd.read_file("/home/jovyan/work/GGIS 407/Final Project/data/ice-aor-county-shp.shp")

In [19]:
aor.head()

,STATEFP,COUNTYF,COUNTYN,GEOIDFQ,GEOID,NAME,NAMELSA,STUSPS,STATE_N,LSAD,ALAND,AWATER,offc_nm,notes,aor_nam,geometry
0,06,079,00277304,0500000US06079,06079,San Luis Obispo,San Luis Obispo County,CA,California,06,8.549141e+09,815650229.0,Los Angeles,None,Los Angeles,"POLYGON ((-121.34636 35.79518, -121.24498 35.7..."
1,06,081,00277305,0500000US06081,06081,San Mateo,San Mateo County,CA,California,06,1.161921e+09,757163219.0,San Francisco,Counties in Northern California not in the LA ...,San Francisco,"POLYGON ((-122.52085 37.59418, -122.51533 37.5..."
2,06,091,00277310,0500000US06091,06091,Sierra,Sierra County,CA,California,06,2.468695e+09,23299110.0,San Francisco,Counties in Northern California not in the LA ...,San Francisco,"POLYGON ((-121.05748 39.53999, -121.05643 39.5..."
3,06,067,00277298,0500000US06067,06067,Sacramento,Sacramento County,CA,California,06,2.500063e+09,75323439.0,San Francisco,Counties in Northern California not in the LA ...,San Francisco,"POLYGON ((-121.86254 38.06795, -121.86160 38.0..."
4,06,107,00277318,0500000US06107,06107,Tulare,Tulare County,CA,California,06,1.249384e+10,37260863.0,San Francisco,Counties in Northern California not in the LA ...,San Francisco,"POLYGON ((-119.56647 36.49434, -119.56366 36.4..."


In [20]:
print(aor.columns.tolist())
print(aor.crs)

['STATEFP', 'COUNTYF', 'COUNTYN', 'GEOIDFQ', 'GEOID', 'NAME', 'NAMELSA', 'STUSPS', 'STATE_N', 'LSAD', 'ALAND', 'AWATER', 'offc_nm', 'notes', 'aor_nam', 'geometry']
epsg:4269


In [21]:
texas_aor = aor[aor['STATE_N'] == 'Texas'] 

In [22]:
print(texas_aor['aor_nam'].unique())  
print(texas_aor.groupby('aor_nam')['NAMELSA'].count()) 

['Houston' 'Dallas' 'Harlingen' 'El Paso' 'San Antonio']
aor_nam
Dallas         128
El Paso         18
Harlingen       15
Houston         57
San Antonio     36
Name: NAMELSA, dtype: int64


## Find and retrieve nearest nodes for immigration courts, facilities, and towers on OSMnx network graphs:

In [23]:
#courts_gdf
courts = gpd.read_file("/home/jovyan/work/GGIS 407/Final Project/data/processed/courts_gdf.geojson")
detention_centers = gpd.read_file("/home/jovyan/work/GGIS 407/Final Project/data/processed/EOIR_complete_gdf.geojson")
counties_287g_enforcement_types = gpd.read_file("/home/jovyan/work/GGIS 407/Final Project/data/processed/tx_287g_counties_gdf.geojson")                

In [24]:
facilities = pd.read_csv("/home/jovyan/work/GGIS 407/Final Project/data/facilities.csv")
courts = pd.read_csv("/home/jovyan/work/GGIS 407/Final Project/data/texas_immigration_judges.csv")

print(facilities.columns.tolist())
print(courts.columns.tolist())

['detention_facility_code', 'detention_facility_name', 'address', 'city', 'county', 'state', 'zip', 'aor', 'latitude', 'longitude', 'type_detailed', 'type_grouped']
['Court', 'Judge_Name', 'Initials', 'WebEx_Link', 'Access_Code']


## Now get road networks for each AOR within buffer zone of points of interest nodes

In [ ]:
G_drive_harlingen = texas_aor[texas_aor['aor_nam'] == 'Harlingen']
harlingen_union_poly = G_drive_harlingen.geometry.unary_union
G_drive_harlingen = ox.graph_from_polygon(harlingen_union_poly, network_type='drive')
G_drive_harlingen = ox.add_edge_speeds(G_drive_harlingen)
G_drive_harlingen = ox.add_edge_travel_times(G_drive_harlingen)
ox.save_graphml(G_drive_harlingen, filepath="data/G_drive_harlingen.graphml")
print("Done!")

In [ ]:
for aor_name, group in texas_aor.groupby('aor_nam'):
    print(f"Downloading {aor_name}...")
    union_poly = group.geometry.unary_union
    G_drive = ox.graph_from_polygon(union_poly, network_type='drive')
    G_drive = ox.add_edge_speeds(G_drive)
    G_drive = ox.add_edge_travel_times(G_drive)
    ox.save_graphml(G, filepath=f"data/{aor_name.lower().replace('G_',' ', '_')}_drive.graphml")
    print(f"Saved {aor_name}!")

In [ ]:
# Bounding box for area of interest (Texas)


# Download a drivable street network from OpenStreetMap
G_drive = ox.graph_from_place("Texas, USA", network_type='drive')
G_drive_time = ox.add_edge_speeds(G_drive)
G_drive_time = ox.add_edge_travel_times(G_drive_time)
ox.save_graphml(G_drive_time, filepath="data/G_drive_tx.graphml")